# Crop Doctor: Colab setup (data, once)

Run this notebook **in Google Colab** (Runtime → Change runtime type → **T4 GPU**). It gets every photo the model needs onto
Google's machine, never onto your laptop.

- **First time:** downloads all four sources (PlantVillage, PlantDoc, Tomato-Village, Tanzania potato), about
  10 GB of downloads and 15–25 minutes, then saves one archive (a few GB) to your Google Drive.
- **Every time after:** restores that archive from Drive in a minute or two.

The last cell checks that every photo listed in `crop_doctor/data/splits_v3` is present. When it says `0 missing`,
the training and accuracy notebooks can run.

## 1. GPU and Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"
from google.colab import drive
drive.mount("/content/drive")

## 2. Get the code

In [ ]:
import os, subprocess
REPO = "https://github.com/krishna-chhabra-cse/crop-doctor.git"
CODE = "/content/crop-doctor"
if os.path.exists(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
%cd {CODE}
!pip install -q timm  # Colab already has torch, torchvision, pandas, scikit-learn and pillow

## 3. Photos: restore from Drive, or download once

The photos live on Colab's fast local disk (`/content/datasets`). Drive only keeps one archive of them, because Drive is
slow with thousands of small files.

In [ ]:
from pathlib import Path
LOCAL = Path("/content/datasets")
ARCHIVE = Path("/content/drive/MyDrive/crop_doctor/datasets.tar")   # change the folder if you like
os.environ["CROP_DOCTOR_DATASETS"] = str(LOCAL)

if LOCAL.exists() and any(LOCAL.iterdir()):
    print("Photos already on this machine.")
elif ARCHIVE.exists():
    print(f"Restoring {ARCHIVE.stat().st_size / 1e9:.1f} GB from Drive ...")
    LOCAL.mkdir(parents=True, exist_ok=True)
    !tar -xf "{ARCHIVE}" -C /content
else:
    print("First run: downloading every source (15-25 minutes).")
    !python -m crop_doctor.data.download --all

## 4. Save to Drive (first run only)

In [ ]:
if not ARCHIVE.exists():
    ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    !tar -cf "{ARCHIVE}" -C /content datasets
    print(f"Saved {ARCHIVE.stat().st_size / 1e9:.1f} GB to Drive. Next time this notebook restores it instead.")
else:
    print("Archive already on Drive.")

## 5. Check: every photo the splits need

In [ ]:
!python -m crop_doctor.data.download --check